In [1]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

In [2]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """
    function to get the weather of the current location
    """
    import requests

    loc = requests.get("http://ip-api.com/json/").json()
    weather = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={loc['lat']}&longitude={loc['lon']}&current_weather=true").json()

    return(weather)

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "claude-sonnet-4-6",
    temperature=0.5,
    timeout=10,
    max_tokens=1000
)

In [4]:
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [9]:
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_weather_for_location],
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather like today?"}]},
    config=config
)

print(response['structured_response'])

ResponseFormat(punny_response="Looks like it's a *hot* topic today! 🌞 It's a scorching **33.4°C** with a light breeze of **7.3 km/h** blowing in from the northeast. The skies are mostly clear — so you could say the sun is really *raising the stakes*! Don't forget your sunscreen, or you might end up in a *sticky situation*. Stay cool out there — though the weather clearly didn't get that memo! 😄", weather_conditions='Temperature: 33.4°C, Wind Speed: 7.3 km/h, Wind Direction: 33° (NNE), Mostly Clear Skies, Daytime')
